# Stage B Final Audit

Run this short notebook before manuscript rewriting.

It resolves the remaining **MNB Stage-A2 vs Stage-B discrepancy** and prints compact manuscript-ready diagnostic summaries, including ROC-AUC/PR-AUC and length-control deltas.

In [ ]:
!pip -q install datasets joblib
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, re, hashlib, joblib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

REV=Path("/content/drive/MyDrive/stress_detection_reviewer_revision")
A2=REV/"stageA2"
B=REV/"stageB"
BR=B/"results"
BP=B/"predictions"
BM=B/"models"

## 1. Compare Stage-A and Stage-B selected MNB parameters

In [ ]:
a=pd.read_csv(REV/"mnb_protocol_audit.csv")
a=a[a.protocol=="leakage_clean"].iloc[0]
a_params=json.loads(a.best_params)

b_meta=json.loads((BM/"Multinomial_NB_selection.json").read_text())
print("Stage A params:",a_params)
print("Stage A CV Macro-F1:",a.best_cv_macro_f1)
print("\nStage B params:",b_meta["best_params"])
print("Stage B CV Macro-F1:",b_meta["best_cv_macro_f1"])
print("\nSame parameters:",a_params==b_meta["best_params"])

## 2. Rebuild the current leakage-clean data and independently refit MNB

In [ ]:
def norm(t):
    if pd.isna(t): return ""
    t=str(t).lower().strip()
    t=re.sub(r"https?://\\S+|www\\.\\S+","<url>",t)
    t=re.sub(r"\\s+"," ",t)
    return t

def h(t):
    return hashlib.sha256(norm(t).encode()).hexdigest()

ds=load_dataset("andreagasparini/dreaddit")
tr=ds["train"].to_pandas()[["text","label"]].copy()
te=ds["test"].to_pandas()[["text","label"]].copy()
tr["hash"]=tr.text.map(h)
te["hash"]=te.text.map(h)
ov=set(tr["hash"]) & set(te["hash"])
clean=tr.drop_duplicates("hash",keep="first")
clean=clean[~clean["hash"].isin(ov)].reset_index(drop=True)

fp=hashlib.sha256()
for x,y in zip(clean.text.astype(str),clean.label.astype(int)):
    fp.update(x.encode("utf-8"))
    fp.update(str(y).encode())

print("Clean n:",len(clean))
print("Current clean-data fingerprint:",fp.hexdigest())

alpha=float(a_params["clf__alpha"])
mf=int(a_params["tfidf__max_features"])
vec=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=.98,max_features=mf)
X=vec.fit_transform(clean.text.astype(str))
model=MultinomialNB(alpha=alpha).fit(X,clean.label.astype(int))
pred=model.predict(vec.transform(te.text.astype(str)))
pp=model.predict_proba(vec.transform(te.text.astype(str)))[:,1]

print("Independent current refit:",
      "accuracy",accuracy_score(te.label,pred),
      "macro_f1",f1_score(te.label,pred,average="macro"),
      "auc",roc_auc_score(te.label,pp))

## 3. Determine which saved stream the independent refit matches

In [ ]:
cur=pred
rows=[]
for label,path in [
    ("Stage A2",A2/"MNB_CANONICAL_Dreaddit_Test.csv"),
    ("Stage B",BP/"Multinomial_NB_Dreaddit_Test.csv")
]:
    d=pd.read_csv(path)
    p=d.predicted_label.to_numpy()
    rows.append({
        "stream":label,
        "n":len(p),
        "disagreements_vs_independent_refit":int(np.sum(p!=cur)),
        "macro_f1":f1_score(d.true_label,p,average="macro"),
        "accuracy":accuracy_score(d.true_label,p)
    })
display(pd.DataFrame(rows))

print("\nDecision rule:")
print("- If Stage B has 0 disagreements, Stage B is the reproducible canonical full-clean protocol.")
print("- If Stage A2 has 0 disagreements, keep Stage A2.")
print("- If neither has 0, stop and send this output before manuscript editing.")

## 4. Manuscript-ready six-model summary

In [ ]:
s=pd.read_csv(BR/"ALL_SIX_MODELS_summary.csv")
cols=["model","dataset","macro_f1_mean","macro_f1_sd","balanced_accuracy_mean",
      "positive_recall_mean","predicted_positive_rate_mean","roc_auc_mean","pr_auc_mean"]
display(s[cols].sort_values(["dataset","macro_f1_mean"],ascending=[True,False]).round(6))

## 5. Mean RoBERTa label-shift sensitivity by dataset

In [ ]:
p=pd.read_csv(BR/"roberta_label_shift_EM_sensitivity.csv")
summ=(p.groupby("dataset")
      .agg(original_macro_f1=("original_macro_f1","mean"),
           adjusted_macro_f1=("adjusted_macro_f1","mean"),
           delta_macro_f1=("delta_macro_f1","mean"),
           estimated_target_prior=("estimated_target_prior_positive_unlabeled","mean"),
           actual_target_prior=("actual_target_prior_positive_for_evaluation_only","mean"),
           original_positive_recall=("original_positive_recall","mean"),
           adjusted_positive_recall=("adjusted_positive_recall","mean"))
      .reset_index())
display(summ.round(6))

## 6. Mean length-control effect

In [ ]:
l=pd.read_csv(BR/"length_control_dreaddit.csv")
agg=(l.groupby(["model","variant"])
     .agg(macro_f1=("macro_f1","mean"),
          positive_recall=("positive_recall","mean"),
          roc_auc=("roc_auc","mean"))
     .reset_index())

wide=agg.pivot(index="model",columns="variant",values=["macro_f1","positive_recall","roc_auc"])
wide[("macro_f1","delta_truncated_minus_original")]=(
    wide[("macro_f1","twitter_length_truncated")]-
    wide[("macro_f1","original")]
)
display(wide.round(6))